# RNNs and LSTMs

Tutorial of Computational Linguistics, National Chengchi University

*Chang-Yu Tsai, 2025.04.11*

- In this week, we will try:
  - to build a model with RNN and its variants
  - to run on the T4 GPU
  - to train embeddings based on our dataset
  


## Set-up

- importing required packages

```
import pandas as pd
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import numpy as np

import re

import jieba

from collections import Counter
```


In [ ]:
import pandas as pd
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import numpy as np

import re

import jieba

from collections import Counter

## Preprocessing

- downloading the dataset from github
> Our data was collected from Cofacts. If you want to explore more on their dataset, please refer to [this website](https://huggingface.co/datasets/Cofacts/line-msg-fact-check-tw).

```
!wget https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/balanced_misinformation.csv
```

In [ ]:
!wget https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/balanced_misinformation.csv

--2025-04-11 03:31:36--  https://raw.githubusercontent.com/EntropiaTsai/nccu_elt_course_material/refs/heads/main/balanced_misinformation.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18379337 (18M) [text/plain]
Saving to: ‘balanced_misinformation.csv.1’

balanced_misinforma 100%[===================>]  17.53M  --.-KB/s    in 0.1s    

2025-04-11 03:31:36 (146 MB/s) - ‘balanced_misinformation.csv.1’ saved [18379337/18379337]



- reading the file

> To save time and memory, we only run the model on 3000 rows of the dataset.

```
df = pd.read_csv('balanced_misinformation.csv', encoding='utf-8')
df_subset=df[:3000]
```

In [ ]:
df = pd.read_csv('balanced_misinformation.csv', encoding='utf-8')
df_subset=df[:3000]

### Text Encoding

- segmentation

```
# conducting segmentation with `jieba`
tokenised_texts = []
for text in df_subset["text"]:
  tokens=list(jieba.cut(text))
  tokenised_texts.append(tokens)

sentence_lengths = []
for sent in tokenised_texts:
  sentence_len=len(sent)
  sentence_lengths.append(sentence_len)

max_len =  max(sentence_lengths)
print("max_len：", max_len)
```

In [ ]:
# conducting segmentation with `jieba`
tokenised_texts = []
for text in df_subset["text"]:
  tokens=list(jieba.cut(text))
  tokenised_texts.append(tokens)

sentence_lengths = []
for sent in tokenised_texts:
  sentence_len=len(sent)
  sentence_lengths.append(sentence_len)

max_len =  max(sentence_lengths)
print("max_len：", max_len)

max_len： 5426


- creating the vocabuary list
<img src="https://hackmd.io/_uploads/Bkj8KGBR1g.jpg" width="80%">

  > We create a list (dictionary) in which each token is mapped to a unique ID.  
  In the model introduced later, these IDs will be further converted into embeddings through an embedding layer.

  - `<PAD>` (padding token):

    Deep learning models (like RNNs and LSTMs) usually expect input sequences to have the same length within a batch.
    Since natural language sentences have variable lengths, we use <PAD> to extend shorter sentences to a fixed length, making all inputs the same shape. Padding ensures that the model can process batches efficiently and in parallel.
    > That's why we calculate the maximum sentence length.

  - `<UNK>` (unknown token):

    During inference or even in training, there might be words in a sentence that were not seen during vocabulary creation (e.g., typos, rare words, or words from new domains).
    To handle these unseen tokens, we reserve a special ID for unknowns: <UNK>.
    This prevents errors during lookup and gives the model a fallback representation.



**💡 What if I have other features?**
>You can also include other sequence-based features, such as POS tags. Similarly, you need to create a list (dictionary) in which each tag is mapped to a unique ID. These IDs will be further converted into embeddings through an embedding layer in the model.

```
all_tokens = []
for sent in tokenised_texts:
  for token in sent:
      all_tokens.append(token)

word_counts = Counter(all_tokens)


vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in word_counts.items():
    vocab[word] = len(vocab)
```

In [ ]:
all_tokens = []
for sent in tokenised_texts:
  for token in sent:
      all_tokens.append(token)

word_counts = Counter(all_tokens)


vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in word_counts.items():
    vocab[word] = len(vocab)



- creating the IDs of each token

```
encoded_inputs = []
for sent in tokenised_texts:
    # converting each token into its ID on the vocabuary list
    ids = []
    for token in sent:
        if token in vocab:
            ids.append(vocab[token])
        else:
            ids.append(vocab["<UNK>"]) # if the token is not in the vocabuary list, it is assigned the ID of `UNK`

    # padding
    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))

    encoded_inputs.append(ids)

print("The input number:")
print(len(encoded_inputs))

print("The first row of the texts is encoded as:")
print(encoded_inputs[0])
```


In [ ]:
encoded_inputs = []
for sent in tokenised_texts:
    # converting each token into its ID on the vocabuary list
    ids = []
    for token in sent:
        if token in vocab:
            ids.append(vocab[token])
        else:
            ids.append(vocab["<UNK>"]) # if the token is not in the vocabuary list, it is assigned the ID of `UNK`

    # padding
    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))

    encoded_inputs.append(ids)

print("The input number:")
print(len(encoded_inputs))

print("The first row of the texts is encoded as:")
print(encoded_inputs[0])

The input number:
3000
The first row of the texts is encoded as:
[2, 3, 4, 4, 5, 6, 7, 6, 8, 6, 9, 4, 10, 11, 12, 13, 14, 15, 16, 11, 17, 18, 11, 19, 20, 11, 21, 22, 23, 24, 25, 26, 27, 28, 29, 14, 15, 22, 23, 14, 30, 31, 27, 32, 33, 11, 34, 30, 14, 17, 35, 14, 36, 37, 38, 18, 12, 11, 39, 40, 38, 41, 42, 24, 43, 44, 27, 44, 45, 46, 47, 6, 48, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### Label encoding

```
# converting labels
labels_label_encoder = LabelEncoder()
df_subset["type_encoded"] = labels_label_encoder.fit_transform(df_subset["type"])
labels = df_subset["type_encoded"].tolist()

print("The label number:")
print(len(labels))
```

In [ ]:
# converting labels
labels_label_encoder = LabelEncoder()
df_subset["type_encoded"] = labels_label_encoder.fit_transform(df_subset["type"])
labels = df_subset["type_encoded"].tolist()

print("The label number:")
print(len(labels))

The label number:
3000


<ipython-input-41-c25ba18d2580>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset["type_encoded"] = labels_label_encoder.fit_transform(df_subset["type"])


### Data splitting

```
# Step 1: 70% for the training set and 30% for the remaining data
feat_train, feat_devtest, label_train, label_devtest = train_test_split(
    encoded_inputs, labels, test_size=0.3, random_state=42
)

# Step 2: in the remaining data, 10% for the dev set and 20% for the test set
feat_dev, feat_test, label_dev, label_test = train_test_split(
    feat_devtest, label_devtest, test_size=2/3, random_state=42
)
```

In [ ]:
# Step 1: 70% for the training set and 30% for the remaining data
feat_train, feat_devtest, label_train, label_devtest = train_test_split(
    encoded_inputs, labels, test_size=0.3, random_state=42
)

# Step 2: in the remaining data, 10% for the dev set and 20% for the test set
feat_dev, feat_test, label_dev, label_test = train_test_split(
    feat_devtest, label_devtest, test_size=2/3, random_state=42
)

### `DataLoader` creating

```
# converting data into tensor
feat_train_tensor = torch.tensor(feat_train, dtype=torch.long)
label_train_tensor = torch.tensor(label_train, dtype=torch.long)

feat_dev_tensor = torch.tensor(feat_dev, dtype=torch.long)
label_dev_tensor = torch.tensor(label_dev, dtype=torch.long)

feat_test_tensor = torch.tensor(feat_test, dtype=torch.long)
label_test_tensor = torch.tensor(label_test, dtype=torch.long)

# creating TensorDataset
train_dataset = TensorDataset(feat_train_tensor, label_train_tensor)
dev_dataset   = TensorDataset(feat_dev_tensor, label_dev_tensor)
test_dataset  = TensorDataset(feat_test_tensor, label_test_tensor)

# creating DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
```

In [ ]:
# converting data into tensor
feat_train_tensor = torch.tensor(feat_train, dtype=torch.long)
label_train_tensor = torch.tensor(label_train, dtype=torch.long)

feat_dev_tensor = torch.tensor(feat_dev, dtype=torch.long)
label_dev_tensor = torch.tensor(label_dev, dtype=torch.long)

feat_test_tensor = torch.tensor(feat_test, dtype=torch.long)
label_test_tensor = torch.tensor(label_test, dtype=torch.long)

# creating TensorDataset
train_dataset = TensorDataset(feat_train_tensor, label_train_tensor)
dev_dataset   = TensorDataset(feat_dev_tensor, label_dev_tensor)
test_dataset  = TensorDataset(feat_test_tensor, label_test_tensor)

# creating DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## RNN



### Model defining


> Note that we use `self.embedding = nn.Embedding(vocab_size, embed_dim)` to obtain embeddings because our input data consists of token IDs.
However, if we use pretrained embeddings such as `Word2Vec`, `GloVe`, or `BERT`, this embedding layer is no longer needed, since the input is already represented as continuous vectors.

```
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)   # converting IDs into embeddings based on the vocab list

        self.rnn = nn.RNN(                                     
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):                  # the shape of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, h_n = self.rnn(embedded)        # the shape of `h_n`: (num_layers, batch_size, hidden_dim)

        final_hidden = h_n[-1]             # last layer's hidden state (forward only)
        logits = self.output(final_hidden) # the shape of `logits`: (batch_size, num_classes)

        return logits
```


In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)   # converting IDs into embeddings based on the vocab list

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):                  # the shape of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, h_n = self.rnn(embedded)        # the shape of `h_n`: (num_layers, batch_size, hidden_dim)

        final_hidden = h_n[-1]             # last layer's hidden state (forward only)
        logits = self.output(final_hidden) # the shape of `logits`: (batch_size, num_classes)

        return logits

- initialising the model

```
torch.manual_seed(4)  # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabulary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within RNN
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of RNN layers

RNNmodel = RNN(                
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    num_layers=num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    RNNmodel.parameters(),    
    lr=learning_rate,
    weight_decay=l2_lambda
)
```

In [ ]:
torch.manual_seed(4)  # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabulary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within RNN
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of RNN layers

RNNmodel = RNN(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    num_layers=num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    RNNmodel.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)

### `GPU` Setting

It is common to run deep learning models on a `GPU` to save both time and memory.  
- It is necessary to specify the `GPU` device on which the model should run; otherwise, the model will run on the `CPU` by default.

```
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RNNmodel = RNNmodel.to(device)
print("We are using",next(RNNmodel.parameters()).device,".")
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RNNmodel = RNNmodel.to(device)
print("We are using",next(RNNmodel.parameters()).device,".")

We are using cuda:0 .


### Training

**❗ Note that we also need to move `batch_feat` and `batch_label` to the same `GPU` device on which the model is running.**

```
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
    RNNmodel.train()  # starting training
    train_loss = 0.0

    for batch_feat, batch_label in train_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        optimizer.zero_grad()
        predictions = RNNmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation phase
    RNNmodel.eval()  # starting evaluating on the dev set
    dev_loss = 0.0

    with torch.no_grad():  # pausing calculating the gradient
        for batch_feat, batch_label in dev_loader:
            batch_feat = batch_feat.to(device).long()    # moving features to the GPU
            batch_label = batch_label.to(device).long()  # moving labels to the GPU
            predictions = RNNmodel(batch_feat)
            loss = criterion(predictions, batch_label)
            dev_loss += loss.item() * batch_feat.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    # saving the value the loss of each epoch
    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")
```


In [ ]:
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
    RNNmodel.train()  # starting training
    train_loss = 0.0

    for batch_feat, batch_label in train_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        optimizer.zero_grad()
        predictions = RNNmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation phase
    RNNmodel.eval()  # starting evaluating on the dev set
    dev_loss = 0.0

    with torch.no_grad():  # pausing calculating the gradient
        for batch_feat, batch_label in dev_loader:
            batch_feat = batch_feat.to(device).long()    # moving features to the GPU
            batch_label = batch_label.to(device).long()  # moving labels to the GPU
            predictions = RNNmodel(batch_feat)
            loss = criterion(predictions, batch_label)
            dev_loss += loss.item() * batch_feat.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    # saving the value the loss of each epoch
    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")

Epoch 1/1 - Train Loss: 1.1124 - Dev Loss: 1.1018


### Predicting and evaluating


```
# prediciting
RNNmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
    for batch_feat, batch_label in test_loader:
        batch_feat = batch_feat.to(device).long()   # moving features to the GPU
        batch_label = batch_label.to(device).long() # moving labels to the GPU

        test_predictions = RNNmodel(batch_feat)
        predicted_labels = torch.argmax(test_predictions, dim=1)

        label_true.extend(batch_label.cpu().numpy())
        label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
```


In [ ]:


# prediciting
RNNmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
    for batch_feat, batch_label in test_loader:
        batch_feat = batch_feat.to(device).long()   # moving features to the GPU
        batch_label = batch_label.to(device).long() # moving labels to the GPU

        test_predictions = RNNmodel(batch_feat)
        predicted_labels = torch.argmax(test_predictions, dim=1)

        label_true.extend(batch_label.cpu().numpy())
        label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")



Test Accuracy: 0.3200
Precision: 0.1024, Recall: 0.3200, F1 Score: 0.1552


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## LSTM



### Model defining

```
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):                  # the shape of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(embedded)  # the shape of `h_n`: (num_layers, batch_size, hidden_dim)

        final_hidden = h_n[-1]             # last layer's hidden state (forward only)
        logits = self.output(final_hidden) # the shape of `logits`: (batch_size, num_classes)

        return logits
```

In [ ]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):                  # the shape of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(embedded)  # the shape of `h_n`: (num_layers, batch_size, hidden_dim)

        final_hidden = h_n[-1]             # last layer's hidden state (forward only)
        logits = self.output(final_hidden) # the shape of `logits`: (batch_size, num_classes)

        return logits

- initialising the model

```
torch.manual_seed(4) # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabuary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within LSTM
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of the LSTM layer

LSTMmodel = LSTM(
    vocab_size = vocab_size,
    embed_dim = embed_dim,
    hidden_dim = hidden_dim,
    num_classes = num_classes,
    num_layers = num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    LSTMmodel.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)
```

In [ ]:
torch.manual_seed(4) # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabuary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within LSTM
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of the LSTM layer

LSTMmodel = LSTM(
    vocab_size = vocab_size,
    embed_dim = embed_dim,
    hidden_dim = hidden_dim,
    num_classes = num_classes,
    num_layers = num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    LSTMmodel.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)

### `GPU` Setting

It is common to run deep learning models on a `GPU` to save both time and memory.  
- It is necessary to specify the `GPU` device on which the model should run; otherwise, the model will run on the `CPU` by default.

```
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LSTMmodel = LSTMmodel.to(device)
print("We are using",next(LSTMmodel.parameters()).device)
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LSTMmodel = LSTMmodel.to(device)
print("We are using",next(LSTMmodel.parameters()).device)

We are using cuda:0


### Training

**❗ Note that we also need to move `batch_feat` and `batch_label` to the same `GPU` device on which the model is running.**

```
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
    LSTMmodel.train()  # starting training
    train_loss = 0.0

    for batch_feat, batch_label in train_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        optimizer.zero_grad()
        predictions = LSTMmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation phase
    LSTMmodel.eval()  # starting evaluating on the dev set
    dev_loss = 0.0

    with torch.no_grad():  # pausing calculating the gradient
        for batch_feat, batch_label in dev_loader:
            batch_feat = batch_feat.to(device).long()    # moving features to the GPU
            batch_label = batch_label.to(device).long()  # moving labels to the GPU
            predictions = LSTMmodel(batch_feat)
            loss = criterion(predictions, batch_label)
            dev_loss += loss.item() * batch_feat.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    # saving the value the loss of each epoch
    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")
```


In [ ]:
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
    LSTMmodel.train()  # starting training
    train_loss = 0.0

    for batch_feat, batch_label in train_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        optimizer.zero_grad()
        predictions = LSTMmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

    avg_train_loss = train_loss / len(train_loader.dataset)

    # validation phase
    LSTMmodel.eval()  # starting evaluating on the dev set
    dev_loss = 0.0

    with torch.no_grad():  # pausing calculating the gradient
        for batch_feat, batch_label in dev_loader:
            batch_feat = batch_feat.to(device).long()    # moving features to the GPU
            batch_label = batch_label.to(device).long()  # moving labels to the GPU
            predictions = LSTMmodel(batch_feat)
            loss = criterion(predictions, batch_label)
            dev_loss += loss.item() * batch_feat.size(0)

    avg_dev_loss = dev_loss / len(dev_loader.dataset)

    # saving the value the loss of each epoch
    train_losses.append(avg_train_loss)
    dev_losses.append(avg_dev_loss)

    # printing the loss of each epoch
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")

Epoch 1/1 - Train Loss: 1.1058 - Dev Loss: 1.1044


### Predicting and evaluating


```
# prediciting
LSTMmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
    for batch_feat, batch_label in test_loader:
        batch_feat = batch_feat.to(device).long()   # moving features to the GPU
        batch_label = batch_label.to(device).long() # moving labels to the GPU

        test_predictions = LSTMmodel(batch_feat)
        predicted_labels = torch.argmax(test_predictions, dim=1)

        label_true.extend(batch_label.cpu().numpy())
        label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
```


In [ ]:


# prediciting
LSTMmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
    for batch_feat, batch_label in test_loader:
        batch_feat = batch_feat.to(device).long()   # moving features to the GPU
        batch_label = batch_label.to(device).long() # moving labels to the GPU

        test_predictions = LSTMmodel(batch_feat)
        predicted_labels = torch.argmax(test_predictions, dim=1)

        label_true.extend(batch_label.cpu().numpy())
        label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")



Test Accuracy: 0.3200
Precision: 0.1024, Recall: 0.3200, F1 Score: 0.1552


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## BiLSTM



### Model defining
In a `BiLSTM` model, we need to:

1. Set `bidirectional=True` to enable bidirectional processing.  
   This allows the model to read the input sequence both from left to right (forward) and from right to left (backward).

2. Concatenate the final hidden states from both the forward and backward layers.  
   The combined representation captures information from both directions, which is often useful for tasks where context matters on both sides.

```
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)     # converting IDs into embeddings based on the vocab list

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        # Since we are using a bidirectional LSTM,
        # the final hidden state is the concatenation of forward and backward states,
        # so the input size of the output layer should be `hidden_dim * 2`
        self.output = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):                  # the shapr of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(embedded)  # the shape of `h_n`: (num_layers * num_directions, batch_size, hidden_dim)

        # If `num_layers = 2` and `bidirectional = True`, then:
        # h_n = [layer0_forward, layer0_backward, layer1_forward, layer1_backward]
        forward_final = h_n[-2]        # last layer's forward hidden state
        backward_final = h_n[-1]       # last layer's backward hidden state

        final_hidden = torch.cat((forward_final, backward_final), dim=1)  # (batch_size, hidden_dim * 2)

        logits = self.output(final_hidden)  # (batch_size, num_classes)
        return logits
```

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)     # converting IDs into embeddings based on the vocab list

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        # Since we are using a bidirectional LSTM,
        # the final hidden state is the concatenation of forward and backward states,
        # so the input size of the output layer should be `hidden_dim * 2`
        self.output = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):                  # the shapr of `x`: (batch_size, seq_len)
        embedded = self.embedding(x)       # the shape of `embedded`: (batch_size, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(embedded)  # the shape of `h_n`: (num_layers * num_directions, batch_size, hidden_dim)

        # If `num_layers = 2` and `bidirectional = True`, then:
        # h_n = [layer0_forward, layer0_backward, layer1_forward, layer1_backward]
        forward_final = h_n[-2]        # last layer's forward hidden state
        backward_final = h_n[-1]       # last layer's backward hidden state

        final_hidden = torch.cat((forward_final, backward_final), dim=1)  # (batch_size, hidden_dim * 2)

        logits = self.output(final_hidden)  # (batch_size, num_classes)
        return logits

- initialising the model

```
torch.manual_seed(4) # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabuary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within BiLSTM
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of the BiLSTM layer

BiLSTMmodel = BiLSTM(
    vocab_size = vocab_size,
    embed_dim = embed_dim,
    hidden_dim = hidden_dim,
    num_classes = num_classes,
    num_layers = num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    BiLSTMmodel.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)
```

In [ ]:
torch.manual_seed(4) # the random seed of initialisation

vocab_size = len(vocab)        # the size of the vocabuary list
embed_dim = 200                # the dimension of embeddings
hidden_dim = 64                # the dimension of the hidden layer within BiLSTM
num_classes = 3                # the number of the classes
num_layers = 1                 # the number of the BiLSTM layer

BiLSTMmodel = BiLSTM(
    vocab_size = vocab_size,
    embed_dim = embed_dim,
    hidden_dim = hidden_dim,
    num_classes = num_classes,
    num_layers = num_layers
)

# setting the loss function
criterion = nn.CrossEntropyLoss()

# setting up the optimiser
learning_rate = 0.001
l2_lambda = 0.01

optimizer = optim.Adam(
    BiLSTMmodel.parameters(),
    lr=learning_rate,
    weight_decay=l2_lambda
)

### `GPU` Setting

It is common to run deep learning models on a `GPU` to save both time and memory.  
- It is necessary to specify the `GPU` device on which the model should run; otherwise, the model will run on the `CPU` by default.

```
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BiLSTMmodel = BiLSTMmodel.to(device)
print("We are using",next(BiLSTMmodel.parameters()).device)
```


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BiLSTMmodel = BiLSTMmodel.to(device)
print("We are using",next(BiLSTMmodel.parameters()).device)

We are using cuda:0


### Training

**❗ Note that we also need to move `batch_feat` and `batch_label` to the same `GPU` device on which the model is running.**

```
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
  BiLSTMmodel.train()  # starting training
  train_loss = 0.0

  for batch_feat, batch_label in train_loader:
    batch_feat = batch_feat.to(device).long()    # moving features to the GPU
    batch_label = batch_label.to(device).long()  # moving labels to the GPU
    optimizer.zero_grad()
    predictions = BiLSTMmodel(batch_feat)
    loss = criterion(predictions, batch_label)
    loss.backward()
    optimizer.step()

    train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

  avg_train_loss = train_loss / len(train_loader.dataset)

  # validation phase
  BiLSTMmodel.eval()  # starting evaluating on the dev set
  dev_loss = 0.0

  with torch.no_grad():  # pausing calculating the gradient
    for batch_feat, batch_label in dev_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        predictions = BiLSTMmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        dev_loss += loss.item() * batch_feat.size(0)

  avg_dev_loss = dev_loss / len(dev_loader.dataset)

  # saving the value the loss of each epoch
  train_losses.append(avg_train_loss)
  dev_losses.append(avg_dev_loss)

  # printing the loss of each epoch
  print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")
```


In [ ]:
num_epochs = 1
train_losses = []
dev_losses = []

for epoch in range(num_epochs):
  BiLSTMmodel.train()  # starting training
  train_loss = 0.0

  for batch_feat, batch_label in train_loader:
    batch_feat = batch_feat.to(device).long()    # moving features to the GPU
    batch_label = batch_label.to(device).long()  # moving labels to the GPU
    optimizer.zero_grad()
    predictions = BiLSTMmodel(batch_feat)
    loss = criterion(predictions, batch_label)
    loss.backward()
    optimizer.step()

    train_loss += loss.item() * batch_feat.size(0)  # accumulating the loss of each mini-batch

  avg_train_loss = train_loss / len(train_loader.dataset)

  # validation phase
  BiLSTMmodel.eval()  # starting evaluating on the dev set
  dev_loss = 0.0

  with torch.no_grad():  # pausing calculating the gradient
    for batch_feat, batch_label in dev_loader:
        batch_feat = batch_feat.to(device).long()    # moving features to the GPU
        batch_label = batch_label.to(device).long()  # moving labels to the GPU
        predictions = BiLSTMmodel(batch_feat)
        loss = criterion(predictions, batch_label)
        dev_loss += loss.item() * batch_feat.size(0)

  avg_dev_loss = dev_loss / len(dev_loader.dataset)

  # saving the value the loss of each epoch
  train_losses.append(avg_train_loss)
  dev_losses.append(avg_dev_loss)

  # printing the loss of each epoch
  print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Dev Loss: {avg_dev_loss:.4f}")

Epoch 1/1 - Train Loss: 1.1038 - Dev Loss: 1.1003


### Predicting and evaluating


```
# prediciting
BiLSTMmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
  for batch_feat, batch_label in test_loader:
    batch_feat = batch_feat.to(device).long()   # moving features to the GPU
    batch_label = batch_label.to(device).long() # moving labels to the GPU

    test_predictions = BiLSTMmodel(batch_feat)
    predicted_labels = torch.argmax(test_predictions, dim=1)

    label_true.extend(batch_label.cpu().numpy())
    label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
```


In [ ]:


# prediciting
BiLSTMmodel.eval()
label_true = []
label_pred = []

with torch.no_grad():
  for batch_feat, batch_label in test_loader:
    batch_feat = batch_feat.to(device).long()   # moving features to the GPU
    batch_label = batch_label.to(device).long() # moving labels to the GPU

    test_predictions = BiLSTMmodel(batch_feat)
    predicted_labels = torch.argmax(test_predictions, dim=1)

    label_true.extend(batch_label.cpu().numpy())
    label_pred.extend(predicted_labels.cpu().numpy())

# evaluating
accuracy = accuracy_score(label_true, label_pred)
precision = precision_score(label_true, label_pred, average='weighted')
recall = recall_score(label_true, label_pred, average='weighted')
f1 = f1_score(label_true, label_pred, average='weighted')

# printing out the results
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")



Test Accuracy: 0.3800
Precision: 0.4561, Recall: 0.3800, F1 Score: 0.3103


## Extracting embeddings trained by `BiLSTM`

```
embedding_matrix = BiLSTMmodel.embedding.weight.detach().cpu().numpy()
print(embedding_matrix.shape)
output_path = "BiLSTM_trained_embeddings.txt"

# creating the list of mapping index to word
index2word = {}

for word, idx in vocab.items():
  index2word[idx] = word

# writting `txt`
with open(output_path, "w", encoding="utf-8") as f:
  vocab_size, embed_dim = embedding_matrix.shape
  f.write(f"{vocab_size} {embed_dim}\n")  # the header

  for idx in range(vocab_size):
    word = index2word.get(idx, f"UNK_{idx}")
    vector = " ".join([f"{val:.6f}" for val in embedding_matrix[idx]])
    f.write(f"{word} {vector}\n")
```

In [ ]:
embedding_matrix = BiLSTMmodel.embedding.weight.detach().cpu().numpy()
print(embedding_matrix.shape)
output_path = "BiLSTM_trained_embeddings.txt"

# creating the list of mapping index to word
index2word = {}

for word, idx in vocab.items():
  index2word[idx] = word

# writting `txt`
with open(output_path, "w", encoding="utf-8") as f:
  vocab_size, embed_dim = embedding_matrix.shape
  f.write(f"{vocab_size} {embed_dim}\n")  # the header

  for idx in range(vocab_size):
    word = index2word.get(idx, f"UNK_{idx}")
    vector = " ".join([f"{val:.6f}" for val in embedding_matrix[idx]])
    f.write(f"{word} {vector}\n")

(59349, 200)


# Assignment

Please create a new `.ipynb` file to complete your assignment. Don't forget to include your name and relevant information at the top of your code.


1. `BiLSTM`

  - Adjust the hyperparameters and train a new `BiLSTM` model. Explain the reasons for your adjustments. (20%)
  - Compare its performance with the `BiLSTM` model used in class. Explain the differences. (20%)

2. `BiRNN`

  - Using the same hyperparameters from Q1, train a `BiRNN` model on the same dataset. (30%)  
    *(Hint: You might need to refer to the structure of the three models covered in class.)*

  - Compare the performance with the `BiLSTM` model you trained in Q1. Explain the differences. (20%)


**Bonus1:** Train a stacked model. You can select from `RNN`, `BiRNN`, `LSTM` and `BiLSTM`.

**Bonus2:** Use the embeddings trained by your `BiRNN`/`BiLSTM` model to calculate cosine similarities of five words which you selected.
